In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

token = os.environ.get("GEMINI_API_KEY")

if token:
    masked = f"{token[:4]}...{token[-4:]}" if len(token) > 8 else "****"
    print(f"GEMINI_API_KEY loaded ({len(token)} characters): {masked}")
else:
    print("GEMINI_API_KEY not found. Check that .env exists in this directory, "
          "the key is spelled exactly 'GEMINI_API_KEY', and load_dotenv() ran without error.")

GEMINI_API_KEY loaded (53 characters): AQ.A...EMNg


In [3]:
"""
Phase 2 triple extraction: Gemini models only.

Runs the Phase 2 extraction prompt against a fixed sample of chunks for:
    - gemini-3.1-pro-preview
    - gemini-3.6-flash

Produces two outputs:
    - phase2_gemini_raw.xlsx       one row per (model, chunk) with raw JSON
    - phase2_gemini_triples.xlsx   one row per extracted triple, flattened

Requires a .env file (not committed, not shared) with:
    GEMINI_API_KEY=...

Usage:
    python run_phase2_gemini.py
"""

import os
import re
import json
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

load_dotenv()

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise RuntimeError("GEMINI_API_KEY not found. Add it to your .env file.")

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

CHUNKS_PATH = r"C:\Users\olagunju\OneDrive\KSU PROJECT\SHOLA_KSU_PUBLISHED_PAPERS\Data-Minning\NER-PROJECT\PDF_PREPROCESSING_INTO_CHUNKS\chunks_all.parquet"
PROMPT_PATH = "phase2_extraction_prompt.md"
SAMPLE_PATH = "sampled_30_chunks.csv"  # reuse the same 30 chunks as before
N_CHUNKS = 30
RANDOM_SEED = 42
OUTPUT_DIR = Path("model_comparison_output")
OUTPUT_DIR.mkdir(exist_ok=True)

# Pricing per 1M tokens (input, output), USD, as of Aug 2026.
MODEL_PRICING = {
    "gemini-3.1-pro-preview": (2.00, 12.00),
    "gemini-3.6-flash": (1.50, 7.50),
}

GEMINI_MODELS = ["gemini-3.1-pro-preview", "gemini-3.6-flash"]

# Retry settings for transient 503 "model overloaded" errors, which showed
# up repeatedly in the first run, especially on gemini-3.1-pro-preview.
MAX_RETRIES = 4
RETRY_BACKOFF_SEC = 15  # doubles each retry: 15, 30, 60, 120


# ---------------------------------------------------------------------------
# Load prompt + chunk sample
# ---------------------------------------------------------------------------

def load_system_prompt(path: str) -> str:
    text = Path(path).read_text()
    text = re.sub(r"^#\s+Phase 2.*\n", "", text, count=1)
    return text.strip()


def get_chunk_sample() -> pd.DataFrame:
    """Reuse the exact same 30 chunks from the first run if the CSV is
    available, so this run stays comparable to the OpenAI run. Falls back
    to re-sampling with the same seed if the CSV isn't present.
    """
    if Path(SAMPLE_PATH).exists():
        return pd.read_csv(SAMPLE_PATH)
    df = pd.read_parquet(CHUNKS_PATH)
    df = df[df["chunk_text"].str.strip().str.len() > 200].reset_index(drop=True)
    return df.sample(n=N_CHUNKS, random_state=RANDOM_SEED).reset_index(drop=True)


SYSTEM_PROMPT = load_system_prompt(PROMPT_PATH)
sample_df = get_chunk_sample()
print(f"Using {len(sample_df)} chunks from {SAMPLE_PATH if Path(SAMPLE_PATH).exists() else 'fresh sample'}")


def build_user_message(row: pd.Series) -> str:
    return (
        f"pmid: {row['doi']}\n"
        f"section: {row['section']}\n\n"
        f"chunk_text:\n{row['chunk_text']}"
    )


# ---------------------------------------------------------------------------
# JSON parsing helper
# ---------------------------------------------------------------------------

def parse_triples(raw_text: str):
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw_text.strip(), flags=re.MULTILINE)
    try:
        parsed = json.loads(cleaned)
        if isinstance(parsed, dict):
            parsed = [parsed]
        return parsed, None
    except json.JSONDecodeError as e:
        return None, f"JSON parse error: {e}"


# ---------------------------------------------------------------------------
# Gemini runner with retry on 503 (model overloaded)
# ---------------------------------------------------------------------------

def run_gemini_model(model_name: str, df: pd.DataFrame) -> pd.DataFrame:
    from google import genai
    from google.genai import types

    client = genai.Client(api_key=GEMINI_API_KEY)
    rows = []

    for i, row in df.iterrows():
        user_msg = build_user_message(row)
        attempt = 0
        while True:
            attempt += 1
            start = time.time()
            try:
                response = client.models.generate_content(
                    model=model_name,
                    contents=user_msg,
                    config=types.GenerateContentConfig(
                        system_instruction=SYSTEM_PROMPT,
                        temperature=0,
                    ),
                )
                elapsed = time.time() - start
                raw_output = response.text
                usage = getattr(response, "usage_metadata", None)
                input_tokens = usage.prompt_token_count if usage else None
                output_tokens = usage.candidates_token_count if usage else None
                triples, err = parse_triples(raw_output)

                rows.append({
                    "model": model_name,
                    "chunk_id": row["id"],
                    "doi": row["doi"],
                    "section": row["section"],
                    "raw_output": raw_output,
                    "n_triples": len(triples) if triples is not None else None,
                    "parse_error": err,
                    "input_tokens": input_tokens,
                    "output_tokens": output_tokens,
                    "latency_sec": round(elapsed, 2),
                })
                print(f"  [{model_name}] {i+1}/{len(df)} ok "
                      f"({len(triples) if triples else 0} triples, {elapsed:.1f}s)")
                break

            except Exception as e:
                is_503 = "503" in str(e) or "UNAVAILABLE" in str(e)
                if is_503 and attempt <= MAX_RETRIES:
                    wait = RETRY_BACKOFF_SEC * (2 ** (attempt - 1))
                    print(f"  [{model_name}] {i+1}/{len(df)} overloaded, "
                          f"retry {attempt}/{MAX_RETRIES} in {wait}s...")
                    time.sleep(wait)
                    continue

                rows.append({
                    "model": model_name,
                    "chunk_id": row["id"],
                    "doi": row["doi"],
                    "section": row["section"],
                    "raw_output": None,
                    "n_triples": None,
                    "parse_error": f"API error: {e}",
                    "input_tokens": None,
                    "output_tokens": None,
                    "latency_sec": None,
                })
                print(f"  [{model_name}] {i+1}/{len(df)} FAILED after "
                      f"{attempt} attempt(s): {e}")
                break

    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# Cost estimate
# ---------------------------------------------------------------------------

def add_cost_column(df: pd.DataFrame) -> pd.DataFrame:
    def _cost(r):
        if pd.isna(r["input_tokens"]) or pd.isna(r["output_tokens"]):
            return None
        in_price, out_price = MODEL_PRICING[r["model"]]
        return (r["input_tokens"] / 1_000_000 * in_price) + (r["output_tokens"] / 1_000_000 * out_price)
    df["est_cost_usd"] = df.apply(_cost, axis=1)
    return df


# ---------------------------------------------------------------------------
# Flatten raw_output JSON into one row per triple
# ---------------------------------------------------------------------------

TRIPLE_FIELDS = [
    "pmid", "source", "source_type", "material", "interaction", "target",
    "target_type", "compared_property", "reported_value", "claim_status",
    "flagged_phrase", "corresponding_sentence",
]


def flatten_triples(raw_df: pd.DataFrame) -> pd.DataFrame:
    """Explode each chunk-level row's raw_output JSON into one row per triple.
    Chunks that failed the API call or failed to parse are skipped here;
    they're already visible with their error in the raw file.
    """
    flat_rows = []
    for _, row in raw_df.iterrows():
        if pd.isna(row["raw_output"]):
            continue
        triples, err = parse_triples(row["raw_output"])
        if triples is None:
            continue
        for t in triples:
            flat_row = {
                "model": row["model"],
                "chunk_id": row["chunk_id"],
                "doi": row["doi"],
                "section": row["section"],
            }
            for field in TRIPLE_FIELDS:
                flat_row[field] = t.get(field, "")
            flat_rows.append(flat_row)
    return pd.DataFrame(flat_rows)


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    all_results = []
    for model_name in GEMINI_MODELS:
        print(f"\nRunning {model_name} on {len(sample_df)} chunks...")
        result_df = run_gemini_model(model_name, sample_df)
        all_results.append(result_df)

    combined = pd.concat(all_results, ignore_index=True)
    combined = add_cost_column(combined)

    raw_path = OUTPUT_DIR / "phase2_gemini_raw.xlsx"
    combined.to_excel(raw_path, index=False)
    print(f"\nRaw comparison results -> {raw_path}")

    # Flatten into one row per triple, this is the sheet to actually read.
    triples_df = flatten_triples(combined)
    triples_path = OUTPUT_DIR / "phase2_gemini_triples.xlsx"
    triples_df.to_excel(triples_path, index=False)
    print(f"Flattened triples ({len(triples_df)} rows) -> {triples_path}")

    # Summary rollup.
    summary = (
        combined.groupby("model")
        .agg(
            chunks_run=("chunk_id", "count"),
            chunks_failed=("parse_error", lambda x: x.notna().sum()),
            avg_triples_per_chunk=("n_triples", "mean"),
            total_input_tokens=("input_tokens", "sum"),
            total_output_tokens=("output_tokens", "sum"),
            total_est_cost_usd=("est_cost_usd", "sum"),
            avg_latency_sec=("latency_sec", "mean"),
        )
        .reset_index()
    )
    summary["est_cost_per_1000_chunks_usd"] = (
        summary["total_est_cost_usd"] / len(sample_df) * 1000
    )
    summary_path = OUTPUT_DIR / "phase2_gemini_summary.xlsx"
    summary.to_excel(summary_path, index=False)
    print(f"Summary -> {summary_path}")
    print("\n" + summary.to_string(index=False))
    print(
        "\nNote: avg_triples_per_chunk and chunks_failed are volume/parse-success "
        "signals only, not accuracy. Open phase2_gemini_triples.xlsx and read "
        "against the source chunks to judge actual quality."
    )


if __name__ == "__main__":
    main()

Using 30 chunks from fresh sample

Running gemini-3.1-pro-preview on 30 chunks...


  [gemini-3.1-pro-preview] 1/30 ok (18 triples, 104.2s)


  [gemini-3.1-pro-preview] 2/30 ok (20 triples, 102.7s)


  [gemini-3.1-pro-preview] 3/30 ok (27 triples, 150.9s)
  [gemini-3.1-pro-preview] 4/30 overloaded, retry 1/4 in 15s...
  [gemini-3.1-pro-preview] 4/30 overloaded, retry 2/4 in 30s...


  [gemini-3.1-pro-preview] 4/30 ok (10 triples, 86.3s)


...

  [gemini-3.6-flash] 26/30 ok (0 triples, 2.6s)


  [gemini-3.6-flash] 27/30 ok (10 triples, 22.9s)


  [gemini-3.6-flash] 28/30 ok (0 triples, 4.8s)


  [gemini-3.6-flash] 29/30 ok (4 triples, 15.5s)


  [gemini-3.6-flash] 30/30 ok (8 triples, 27.5s)

Raw comparison results -> model_comparison_output\phase2_gemini_raw.xlsx
Flattened triples (428 rows) -> model_comparison_output\phase2_gemini_triples.xlsx
Summary -> model_comparison_output\phase2_gemini_summary.xlsx

                 model  chunks_run  chunks_failed  avg_triples_per_chunk  total_input_tokens  total_output_tokens  total_est_cost_usd  avg_latency_sec  est_cost_per_1000_chunks_usd
gemini-3.1-pro-preview          30              0                   10.6            168777.0              63091.0            1.094646       100.462667                       36.4882
      gemini-3.6-flash          30              8                    5.0            124264.0              21520.0            0.347796        18.394545                       11.5932

Note: avg_triples_per_chunk and chunks_failed are volume/parse-success signals only, not accuracy. Open phase2_gemini_triples.xlsx and read against the source chunks to judge actual quali